In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from eda_package.data import load_raw_data, clean_data, temporal_split_v2, temporal_split, split_X_y
from eda_package.features import engineer_features
from eda_package.preprocessor import (
    group_countries,
    get_feature_lists,
    create_preprocessor,
    fit_transform_preprocessor,
    preprocess_pipeline,
    transform_preprocessor
)
from eda_package.registry import ORDINAL_FEATURES_MAP, COUNTRY_LIMIT, LEAKY_COLS
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.metrics import classification_report
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

df = load_raw_data()
df = clean_data(df)

#SHOULD WE MOVE THIS TO FEATURES OR DATA?
df = group_countries(df, COUNTRY_LIMIT)
df = engineer_features(df)
df.drop(['country'], axis = 1, inplace = True)

scenario = 2

if scenario == 1:
    training_set, test_set = temporal_split(df, 2017)
    X_train, y_train= split_X_y(training_set)
    X_test, y_test= split_X_y(test_set)
else:
    drop_cols = LEAKY_COLS
    X = df.drop(columns=drop_cols)
    y = df["is_canceled"]
    X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = 0.3,
                                                    random_state = 42)

X_train_processed, X_test_processed = preprocess_pipeline(X_train, X_test)

print(X_train_processed.shape)


(61177, 132)


In [ ]:

model = LogisticRegression(max_iter=1000)
model.fit(X_train_processed, y_train)
y_predicted = model.predict(X_test_processed)

print(f'Accuracy: {round(accuracy_score(y_test, y_predicted),2)}')
print(f'Recall: {round(recall_score(y_test, y_predicted),2)}')
print(f'Precision: {round(precision_score(y_test, y_predicted),2)}')
print(f'F1 score: {round(f1_score(y_test, y_predicted),2)}')


"""
Accuracy: 0.76
Recall: 0.62
Precision: 0.62
F1 score: 0.62
"""

Accuracy: 0.8
Recall: 0.49
Precision: 0.68
F1 score: 0.57


/home/bbedic/.pyenv/versions/3.10.6/envs/noshowshield/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


'\nAccuracy: 0.76\nRecall: 0.62\nPrecision: 0.62\nF1 score: 0.62\n'

In [ ]:


#{'gamma': 10, 'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 100}
final_params = {
    'booster' : 'gbtree',
    'n_estimators': 300, #100, 300
    'max_depth': 10, #3, 10
    'learning_rate': 0.05, #0.2, 0.05
    'gamma': 1, #10, 1
#    'lambda': 1,
#    'alpha': 0,
    'subsample': 0.5, #minimal impact
    'colsample_bytree': 0.3, #minimal impact
    'min_child_weight': 2, #minimal impact
    'random_state': 0,
    'scale_pos_weight': 2, #impact
    'eval_metric': 'logloss'
}
estimator3=XGBClassifier(**final_params)

estimator3.fit(X_train_processed,y_train)
y_pre=estimator3.predict(X_test_processed)

print(f'Accuracy: {round(accuracy_score(y_test, y_pre),2)}')
print(f'Recall: {round(recall_score(y_test, y_pre),2)}')
print(f'Precision: {round(precision_score(y_test, y_pre),2)}')
print(f'F1 score: {round(f1_score(y_test, y_pre),2)}')
print(classification_report(y_test,y_pre))

"""
Accuracy: 0.78
Recall: 0.67
Precision: 0.66
F1 score: 0.66

Accuracy: 0.83
Recall: 0.87
Precision: 0.64
F1 score: 0.74
"""

Accuracy: 0.85
Recall: 0.81
Precision: 0.69
F1 score: 0.74
              precision    recall  f1-score   support

           0       0.92      0.86      0.89     19024
           1       0.69      0.81      0.74      7195

    accuracy                           0.85     26219
   macro avg       0.80      0.83      0.82     26219
weighted avg       0.86      0.85      0.85     26219



'\nAccuracy: 0.78\nRecall: 0.67\nPrecision: 0.66\nF1 score: 0.66\n\nAccuracy: 0.83\nRecall: 0.87\nPrecision: 0.64\nF1 score: 0.74\n'

In [2]:
xgb = XGBClassifier(
    objective = 'binary:logistic',
    booster = 'gbtree',
    subsample = 0.5, #minimal impact
    colsample_bytree = 0.5, #minimal impact
    min_child_weight = 3, #minimal impact
    random_state = 0,
    scale_pos_weight = 3, #impact
    eval_metric = 'logloss'
)


params_dict={'n_estimators': [300,500,700],
    'max_depth': [7, 10, 12],
    'gamma': [0.1, 0.3, 0.5, 0.75, 1],
    'learning_rate': [0.01, 0.03, 0.05, 0.1]
}
grid_model = GridSearchCV(
    xgb,
    param_grid=params_dict,
    scoring='f1',
    cv=5,
    n_jobs=-1
    )

grid_model.fit(X_train_processed,y_train, verbose=True)
best_core_params=grid_model.best_params_
print(best_core_params)

estimator= XGBClassifier(**best_core_params)
estimator.fit(X_train_processed,y_train)
y_predicted=estimator.predict(X_test_processed)

print(f'Accuracy: {round(accuracy_score(y_test, y_predicted),2)}')
print(f'Recall: {round(recall_score(y_test, y_predicted),2)}')
print(f'Precision: {round(precision_score(y_test, y_predicted),2)}')
print(f'F1 score: {round(f1_score(y_test, y_predicted),2)}')


{'gamma': 1, 'learning_rate': 0.03, 'max_depth': 12, 'n_estimators': 500}
Accuracy: 0.86
Recall: 0.69
Precision: 0.76
F1 score: 0.72


In [4]:
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

final_params = {
    'objective' : 'binary:logistic',
    'booster' : 'gbtree',
    'n_estimators': 500,
    'max_depth': 12,
    'learning_rate': 0.03,
    'gamma': 1,
#    'lambda': 1,
#    'alpha': 0,
    'subsample': 0.5, #minimal impact
    'colsample_bytree': 0.5, #minimal impact
    'min_child_weight': 3, #minimal impact
    'random_state': 0,
    'scale_pos_weight': 3, #impact
    'eval_metric': 'logloss'
}
estimator3=XGBClassifier(**final_params)

estimator3.fit(X_train_processed,y_train)
y_pre=estimator3.predict(X_test_processed)

print(f'Accuracy: {round(accuracy_score(y_test, y_pre),2)}')
print(f'Recall: {round(recall_score(y_test, y_pre),2)}')
print(f'Precision: {round(precision_score(y_test, y_pre),2)}')
print(f'F1 score: {round(f1_score(y_test, y_pre),2)}')
print(classification_report(y_test,y_pre))

"""
Accuracy: 0.78
Recall: 0.67
Precision: 0.66
F1 score: 0.66

Accuracy: 0.83
Recall: 0.87
Precision: 0.64
F1 score: 0.74

Accuracy: 0.85
Recall: 0.8
Precision: 0.69
F1 score: 0.74

{'gamma': 0.5, 'learning_rate': 0.05, 'max_depth': 10, 'n_estimators': 500}
Accuracy: 0.84
Recall: 0.85
Precision: 0.66
F1 score: 0.74
"""

Accuracy: 0.84
Recall: 0.85
Precision: 0.66
F1 score: 0.74
              precision    recall  f1-score   support

           0       0.94      0.83      0.88     19024
           1       0.66      0.85      0.74      7195

    accuracy                           0.84     26219
   macro avg       0.80      0.84      0.81     26219
weighted avg       0.86      0.84      0.84     26219



"\nAccuracy: 0.78\nRecall: 0.67\nPrecision: 0.66\nF1 score: 0.66\n\nAccuracy: 0.83\nRecall: 0.87\nPrecision: 0.64\nF1 score: 0.74\n\nAccuracy: 0.85\nRecall: 0.8\nPrecision: 0.69\nF1 score: 0.74\n\n{'gamma': 0.5, 'learning_rate': 0.05, 'max_depth': 10, 'n_estimators': 500}\nAccuracy: 0.84\nRecall: 0.85\nPrecision: 0.66\nF1 score: 0.74\n"

In [5]:
estimator.get_params()

{'objective': 'binary:logistic',
 'base_score': None,
 'booster': None,
 'callbacks': None,
 'colsample_bylevel': None,
 'colsample_bynode': None,
 'colsample_bytree': None,
 'device': None,
 'early_stopping_rounds': None,
 'enable_categorical': False,
 'eval_metric': None,
 'feature_types': None,
 'feature_weights': None,
 'gamma': 1,
 'grow_policy': None,
 'importance_type': None,
 'interaction_constraints': None,
 'learning_rate': 0.03,
 'max_bin': None,
 'max_cat_threshold': None,
 'max_cat_to_onehot': None,
 'max_delta_step': None,
 'max_depth': 12,
 'max_leaves': None,
 'min_child_weight': None,
 'missing': nan,
 'monotone_constraints': None,
 'multi_strategy': None,
 'n_estimators': 500,
 'n_jobs': None,
 'num_parallel_tree': None,
 'random_state': None,
 'reg_alpha': None,
 'reg_lambda': None,
 'sampling_method': None,
 'scale_pos_weight': None,
 'subsample': None,
 'tree_method': None,
 'validate_parameters': None,
 'verbosity': None}

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#model = LogisticRegression(max_iter=10000)
model = SVC(kernel='rbf', C=1, verbose=True)

model.fit(X_train_processed, y_train)
y_predicted = model.predict(X_test_processed)

print(f'Accuracy: {round(accuracy_score(y_test, y_predicted),2)}')
print(f'Recall: {round(recall_score(y_test, y_predicted),2)}')
print(f'Precision: {round(precision_score(y_test, y_predicted),2)}')
print(f'F1 score: {round(f1_score(y_test, y_predicted),2)}')

"""
linear:
Accuracy: 0.8
Recall: 0.48
Precision: 0.69
F1 score: 0.57
"""


[LibSVM].....